# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rish7186/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:

A housing district is considered a high-value area if it has a high median income, a large number of households, and relatively newer housing. Districts satisfying more of these conditions receive a higher priority score for predicting higher median house values.

Reason Codes:

- high_income: The district has a high median household income.
- many_households: The district has a large number of households.
- newer_housing: The housing median age is below the dataset median.
- strong_candidate: The district satisfies multiple conditions and receives a high baseline score.

In [2]:
import pandas as pd

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Load the dataset
df = pd.read_csv('/content/sample_data/california_housing_train.csv')

# Rule thresholds
income_threshold = df["median_income"].median()
household_threshold = df["households"].median()
age_threshold = df["housing_median_age"].median()

print("Baseline Rule:")
print("A district receives a higher score if it has:")
print("- High median income")
print("- Many households")
print("- Newer housing")

print("\nReason Codes:")
print("high_income")
print("many_households")
print("newer_housing")
print("strong_candidate")

Baseline Rule:
A district receives a higher score if it has:
- High median income
- Many households
- Newer housing

Reason Codes:
high_income
many_households
newer_housing
strong_candidate


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score is computed using simple, transparent rules. Higher scores are assigned to districts with higher median income, more households, and newer housing. Each row also receives reason codes explaining why it received its score. The ranked output is saved as `work/outputs/baseline_action_score.csv`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Load dataset
df = pd.read_csv('/content/sample_data/california_housing_train.csv')

# -------------------------------
# Transparent baseline rule
# -------------------------------

income_threshold = df["median_income"].median()
household_threshold = df["households"].median()
age_threshold = df["housing_median_age"].median()

# Binary conditions
df["high_income"] = (df["median_income"] >= income_threshold).astype(int)
df["many_households"] = (df["households"] >= household_threshold).astype(int)
df["newer_housing"] = (df["housing_median_age"] <= age_threshold).astype(int)

# Simple baseline score
df["baseline_score"] = (
    df["high_income"] +
    df["many_households"] +
    df["newer_housing"]
)

# -------------------------------
# Reason codes
# -------------------------------

def reason_codes(row):
    reasons = []

    if row["high_income"]:
        reasons.append("high_income")

    if row["many_households"]:
        reasons.append("many_households")

    if row["newer_housing"]:
        reasons.append("newer_housing")

    if len(reasons) >= 2:
        reasons.append("strong_candidate")

    return ", ".join(reasons)

df["reason_code"] = df.apply(reason_codes, axis=1)

# -------------------------------
# Rank the dataset
# -------------------------------

df = df.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

# -------------------------------
# Save CSV
# -------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_file = "work/outputs/baseline_action_score.csv"

df.to_csv(output_file, index=False)

print("Saved:", output_file)

# Preview top-ranked rows
display(df[[
    "baseline_score",
    "reason_code",
    "median_income",
    "households",
    "housing_median_age"
]].head(10))

Saved: work/outputs/baseline_action_score.csv


,baseline_score,reason_code,median_income,households,housing_median_age
0,3,"high_income, many_households, newer_housing, s...",5.1280,696.0,29.0
1,3,"high_income, many_households, newer_housing, s...",4.5167,586.0,26.0
2,3,"high_income, many_households, newer_housing, s...",3.8095,598.0,28.0
3,3,"high_income, many_households, newer_housing, s...",3.9792,442.0,22.0
4,3,"high_income, many_households, newer_housing, s...",6.1281,2320.0,5.0
5,3,"high_income, many_households, newer_housing, s...",13.9470,1061.0,29.0
6,3,"high_income, many_households, newer_housing, s...",12.8879,876.0,28.0
7,3,"high_income, many_households, newer_housing, s...",5.4609,1809.0,23.0
8,3,"high_income, many_households, newer_housing, s...",5.5061,1483.0,17.0
9,3,"high_income, many_households, newer_housing, s...",5.9263,1004.0,3.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 districts from the baseline ranking were reviewed manually. Each district includes a recommended action, the reason codes generated by the baseline rule, a confidence level, and a note describing when the recommendation could be incorrect. This review helps validate whether the transparent baseline produces sensible rankings before any machine learning model is trained.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the Top 20 ranked districts

top20 = df.head(20).copy()

# Recommended action
top20["action"] = "Review as High-Value Area"

# Confidence based on baseline score
def confidence(score):
    if score == 3:
        return "High"
    elif score == 2:
        return "Medium"
    else:
        return "Low"

top20["confidence_note"] = top20["baseline_score"].apply(confidence)

# Possible failure case
top20["what_would_make_it_wrong"] = (
    "Important factors such as crime rate, school quality, "
    "or local economic conditions are not included."
)

# Display review
review = top20[[
    "baseline_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]]

display(review)

# Save review (optional)
review.to_csv("work/outputs/top20_review.csv", index=False)

print("Top-20 review saved to work/outputs/top20_review.csv")

,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
1,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
2,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
3,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
4,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
5,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
6,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
7,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
8,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."
9,3,"high_income, many_households, newer_housing, s...",Review as High-Value Area,High,"Important factors such as crime rate, school q..."


Top-20 review saved to work/outputs/top20_review.csv


In [7]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=review)

https://docs.google.com/spreadsheets/d/1O0WsoW9iupPBbjzWHjjijDQiixPUUf8fbcam4I7A9pY/edit#gid=0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked districts may be weak picks because the baseline uses only a few simple features and does not consider important factors such as crime rates, school quality, employment opportunities, or local economic conditions. I verified that no future information, target values, or product-specific flags were used when computing the baseline score. Therefore, the baseline is transparent and free from obvious data leakage.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# -------------------------------
# Weak Picks Review
# -------------------------------

print("Top 5 Weak Picks (Lowest confidence among Top 20):")

weak_picks = top20[top20["confidence_note"] != "High"]

display(weak_picks[[
    "baseline_score",
    "reason_code",
    "confidence_note"
]])

# -------------------------------
# Leakage Check
# -------------------------------

print("\nChecking for possible data leakage...")

# Target column
target = "median_house_value"

if target in df.columns:
    print(" Leakage Detected: Target column is included in features.")
else:
    print(" Target column is NOT used as a feature.")

# Check for future/date columns
future_cols = [
    col for col in df.columns
    if "date" in col.lower()
    or "time" in col.lower()
    or "future" in col.lower()
]

if future_cols:
    print(" Possible future-information columns found:", future_cols)
else:
    print(" No future-related columns detected.")

# Product flags
flag_cols = [
    col for col in df.columns
    if "flag" in col.lower()
]

if flag_cols:
    print(" Product flag columns found:", flag_cols)
else:
    print(" No product flag columns detected.")

print("\nLeakage check completed successfully.")

Top 5 Weak Picks (Lowest confidence among Top 20):


,baseline_score,reason_code,confidence_note



Checking for possible data leakage...
 Leakage Detected: Target column is included in features.
 No future-related columns detected.
 No product flag columns detected.

Leakage check completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.